# Notebook 1 — Single cells from Visium HD with **bin2cell**, cell typing with **CellTypist**, and spatial proximity with **squidpy**

### Summer School on Spatial Transcriptomics · France 2026

Human **colorectal cancer (CRC)** Visium HD dataset (10x Genomics).

In this notebook you will:

1. **Reconstruct single cells** from 2 µm Visium HD bins using `bin2cell` (destriping → StarDist nuclear segmentation on H&E → expansion → GEX rescue → bin-to-cell).
2. **Annotate cell types** with `CellTypist`, combining a healthy-gut model with a colorectal-cancer model.
3. **Quantify spatial proximity** between cell types with `squidpy` — neighbourhood enrichment and co-occurrence (this last part is *new*, built on top of the annotated single-cell object).

> **How this notebook is meant to be run in the course.** The full segmentation of a whole Visium HD capture area takes many minutes and a fair amount of RAM/GPU. Every expensive step **saves a checkpoint `.h5ad`** and can be **skipped by loading that checkpoint**. Look for the 💾 *Save* and ⏩ *Load checkpoint* cells. If you are following along live, load the checkpoints; run the heavy cells later on your own machine or on an HPC node.


## 0 · Environment

See `env/environment.yml` in the course repo. Core packages: `bin2cell`, `scanpy`, `celltypist`, `squidpy`, `stardist`, `tensorflow`/`csbdeep`, `opencv-python`.

`bin2cell` will download the StarDist models on first use. `CellTypist` downloads its models with `celltypist.models.download_models(...)`.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc
import bin2cell as b2c
import cv2

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white")
print("scanpy", sc.__version__)
print("bin2cell", b2c.__version__)

## 1 · Data

Download the 10x Genomics **Human Colon Cancer (CRC)** Visium HD dataset. The two inputs `bin2cell` needs are:

* the **2 µm binned output** (`.../square_002um/`), and
* the **high-resolution H&E image** (`..._tissue_image.btf`) that was used as the Space Ranger input.

Run once from a terminal (see also `scripts/download_data.sh`):

```bash
# ~ several GB — do this ahead of the session
mkdir -p data/crc && cd data/crc
curl -O https://cf.10xgenomics.com/samples/spatial-exp/3.0.0/Visium_HD_Human_Colon_Cancer/Visium_HD_Human_Colon_Cancer_binned_outputs.tar.gz
curl -O https://cf.10xgenomics.com/samples/spatial-exp/3.0.0/Visium_HD_Human_Colon_Cancer/Visium_HD_Human_Colon_Cancer_tissue_image.btf
tar -xzf Visium_HD_Human_Colon_Cancer_binned_outputs.tar.gz
```

Then point the two paths below at your download.


In [ ]:
# ---- EDIT THESE PATHS ----
DATA_DIR = "data/crc"                       # where you downloaded the dataset
path = f"{DATA_DIR}/binned_outputs/square_002um/"          # 2um bins
source_image_path = f"{DATA_DIR}/Visium_HD_Human_Colon_Cancer_tissue_image.btf"  # full-res H&E

# where checkpoints are written / read
CKPT_DIR = "data/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

# a working dir for StarDist intermediate images
os.makedirs("stardist", exist_ok=True)

---
# Part A · Reconstruct single cells with bin2cell

`bin2cell` turns the subcellular 2 µm bins into putative **cells** in three ideas:

1. **Destripe** a technical row/column intensity artefact in Visium HD.
2. **Segment nuclei** on a high-resolution H&E image with **StarDist**, and (optionally) segment a gene-expression image to rescue cells with no visible nucleus.
3. **Assign bins to cells** by expanding nuclei and summing expression.


### A.1 · Load and lightly filter

10x moved the bin coordinates into a Parquet file, so `bin2cell` ships a dedicated reader `b2c.read_visium`.

In [ ]:
adata = b2c.read_visium(path, source_image_path=source_image_path)
adata.var_names_make_unique()
adata

In [ ]:
# require genes in >=3 bins, and drop completely empty bins (data is very sparse at 2um)
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.filter_cells(adata, min_counts=1)
adata

### A.2 · Destripe

Visium HD bins vary ~10% in width/height, producing a striped total-count pattern. `b2c.destripe` normalises each row and column by a high quantile of its counts.

In [ ]:
b2c.destripe(adata)

We work with segmentation images at a chosen resolution, set by **`mpp` (microns per pixel)**. Lower `mpp` = sharper image. `mpp≈0.5` works well; small nuclei may benefit from `0.3` or lower.

Let's inspect a small region before/after making the high-res H&E, and check the H&E–grid alignment. If the gene-expression grid is shifted relative to the H&E, nudge it with `up_down` / `right_left`.

In [ ]:
mpp = 0.5
thr = 10  # min adjusted counts to show a bin

# a small region of interest for QC
mask = ((adata.obs['array_row'] >= 2050) & (adata.obs['array_row'] <= 2250) &
        (adata.obs['array_col'] >= 1350) & (adata.obs['array_col'] <= 1550))

bdata = adata[mask].copy()
b2c.scaled_he_image(bdata, mpp=mpp)                 # build a reference H&E just for this region
bdata = bdata[bdata.obs['n_counts_adjusted'] > thr]
sc.set_figure_params(figsize=[5, 5], dpi=100)
sc.pl.spatial(bdata, color=[None, "n_counts_adjusted"], alpha=0.5, cmap='gist_rainbow',
              img_key="0.5_mpp", basis="spatial_cropped")

In [ ]:
# If the coloured grid sits off the tissue, adjust the shift here and re-plot.
right_left = 0    # (-) move image left, (+) right
up_down   = -5    # (-) move image down, (+) up

bdata = adata[mask].copy()
b2c.scaled_he_image(bdata, mpp=mpp)
bdata.obsm['spatial_cropped'][:, 1] += up_down
bdata.obsm['spatial_cropped'][:, 0] += right_left
bdata = bdata[bdata.obs['n_counts_adjusted'] > thr]
sc.set_figure_params(figsize=[5, 5], dpi=200)
sc.pl.spatial(bdata, color=[None, "n_counts_adjusted"], img_key="0.5_mpp",
              basis="spatial_cropped", alpha=0.5, cmap='gist_rainbow')

In [ ]:
# Happy with the shift? Apply it to the full object's spatial coordinates.
adata.obsm['spatial'][:, 1] += up_down
adata.obsm['spatial'][:, 0] += right_left

### A.3 · Build the high-resolution H&E and segment nuclei with StarDist

`b2c.scaled_he_image` writes the working H&E to disk; `b2c.stardist` runs the pre-trained **`2D_versatile_he`** model. We lower `prob_thresh` so the model seeds a generous number of nuclei.

In [ ]:
mpp = 0.3
b2c.scaled_he_image(adata, mpp=mpp, save_path="stardist/he.tiff")

In [ ]:
# H&E nuclear segmentation (this is one of the slower steps)
b2c.stardist(image_path="stardist/he.tiff",
             labels_npz_path="stardist/he.npz",
             stardist_model="2D_versatile_he",
             prob_thresh=0.1)

In [ ]:
# transfer the segmentation labels back onto the bins
b2c.insert_labels(adata,
                  labels_npz_path="stardist/he.npz",
                  basis="spatial",
                  spatial_key="spatial_cropped",
                  mpp=mpp,
                  labels_key="labels_he")

Inspect the nuclear calls on a tiny patch, and use the built-in `b2c.view_stardist_labels` renderer to overlay labels on the actual image.

In [ ]:
patch = ((adata.obs['array_row'] >= 2225) & (adata.obs['array_row'] <= 2275) &
         (adata.obs['array_col'] >= 1400) & (adata.obs['array_col'] <= 1450))

bdata = adata[patch].copy()
bdata = bdata[bdata.obs['labels_he'] > 0]           # 0 == unassigned
bdata.obs['labels_he'] = bdata.obs['labels_he'].astype(str)
sc.pl.spatial(bdata, color=[None, "labels_he"], img_key="0.3_mpp", basis="spatial_cropped")

crop = b2c.get_crop(bdata, basis="spatial", spatial_key="spatial_cropped", mpp=mpp)
rendered = b2c.view_stardist_labels(image_path="stardist/he.tiff",
                                    labels_npz_path="stardist/he.npz", crop=crop)
plt.imshow(rendered); plt.axis('off'); plt.show()

### A.4 · Expand nuclei to whole cells

A nucleus is not a whole cell. `b2c.expand_labels` grows each nucleus outward by up to `max_bin_distance` bins; ties are broken by expression similarity in PCA space.

In [ ]:
b2c.expand_labels(adata,
                  labels_key='labels_he',
                  expanded_labels_key="labels_he_expanded",
                  max_bin_distance=4)

### A.5 · Rescue cells from the gene-expression image (optional but recommended)

Some cells have expression but no clearly segmented nucleus. We build a smoothed **total-counts image**, segment it with the fluorescence StarDist model, and use it only where H&E found nothing (`b2c.salvage_secondary_labels`).

In [ ]:
img = b2c.grid_image(adata, "n_counts_adjusted", mpp=mpp, sigma=5)
cv2.imwrite("stardist/gex.tiff", img)

b2c.stardist(image_path="stardist/gex.tiff",
             labels_npz_path="stardist/gex.npz",
             stardist_model="2D_versatile_fluo",
             prob_thresh=0.01,
             nms_thresh=0.1)

b2c.insert_labels(adata,
                  labels_npz_path="stardist/gex.npz",
                  basis="array",
                  mpp=mpp,
                  labels_key="labels_gex")

In [ ]:
# prefer H&E-expanded labels, fall back to GEX labels where H&E is empty
b2c.salvage_secondary_labels(adata,
                             primary_label="labels_he_expanded",
                             secondary_label="labels_gex",
                             labels_key="labels_joint")

### A.6 · Bin → cell

Finally, sum the bins of each label into a **cell** object. `cdata` is a normal AnnData of cells, ready for downstream single-cell analysis. `.obs['bin_count']` records how many 2 µm bins each cell absorbed.

In [ ]:
cdata = b2c.bin_to_cell(adata, labels_key="labels_joint",
                        spatial_keys=["spatial", "spatial_cropped"])
cdata

In [ ]:
# 💾 Save checkpoint: the reconstructed single-cell object
cdata.write_h5ad(f"{CKPT_DIR}/crc_b2c.h5ad")
# (optional) the full 2um bin object, if you want to revisit segmentation
# adata.write_h5ad(f"{CKPT_DIR}/crc_2um.h5ad")
print("saved", f"{CKPT_DIR}/crc_b2c.h5ad")

---
# Part B · Cell-type annotation with CellTypist

We load the reconstructed cells and annotate them by combining two logistic-regression models:

* **`Human_Colorectal_Cancer`** — a public CellTypist model for CRC (malignant + micro-environment states), and
* *(optional)* a **healthy gut** reference model.

For each cell we keep whichever model is more confident (and is not `Unknown` in the cancer model). This mirrors the reference `bin2cell` CRC analysis.


### ⏩ Load checkpoint (start here if you skipped Part A)

In [ ]:
cdata = sc.read_h5ad(f"{CKPT_DIR}/crc_b2c.h5ad")
cdata.var_names_make_unique()
cdata = cdata[cdata.obs['bin_count'] > 5].copy()   # keep cells with >5 bins
cdata.X.data = np.round(cdata.X.data)              # seurat_v3 HVGs want integers
cdata.raw = cdata.copy()
cdata

In [ ]:
sc.pp.filter_genes(cdata, min_cells=3)
sc.pp.filter_cells(cdata, min_genes=100)
sc.pp.calculate_qc_metrics(cdata, inplace=True)

sc.pp.highly_variable_genes(cdata, n_top_genes=5000, flavor="seurat_v3")
sc.pp.normalize_total(cdata, target_sum=1e4)
sc.pp.log1p(cdata)

In [ ]:
import celltypist
from celltypist import models

# public CRC model (downloads on first use)
models.download_models(model=["Human_Colorectal_Cancer.pkl"], force_update=False)

# OPTIONAL: a healthy-gut model. Set to a local .pkl path to enable the 2-model combine,
# or leave as None to annotate with the CRC model alone.
HEALTHY_GUT_MODEL = None   # e.g. "models/model_from_megaGut_colon_CRC_level3.pkl"

pred_crc = celltypist.annotate(cdata, model="Human_Colorectal_Cancer.pkl", majority_voting=False)
cdata = pred_crc.to_adata()
cdata.obs['predicted_labels_crc'] = cdata.obs['predicted_labels']
cdata.obs['conf_score_crc'] = cdata.obs['conf_score']

In [ ]:
if HEALTHY_GUT_MODEL is not None:
    pred_healthy = celltypist.annotate(cdata, model=HEALTHY_GUT_MODEL, majority_voting=False)
    tmp = pred_healthy.to_adata()
    cdata.obs['predicted_labels_healthy'] = tmp.obs['predicted_labels']
    cdata.obs['conf_score_healthy'] = tmp.obs['conf_score']

    # keep the more confident model per cell; ignore 'Unknown' cancer calls
    higher_in_crc = cdata.obs['conf_score_healthy'] < cdata.obs['conf_score_crc']
    higher_in_crc[cdata.obs['predicted_labels_crc'] == 'Unknown'] = False

    labels = cdata.obs['predicted_labels_healthy'].astype('object')
    labels[higher_in_crc] = cdata.obs.loc[higher_in_crc, 'predicted_labels_crc']
    conf = cdata.obs['conf_score_healthy'].copy()
    conf[higher_in_crc] = cdata.obs.loc[higher_in_crc, 'conf_score_crc']

    cdata.obs['predicted_labels'] = labels.astype('category')
    cdata.obs['conf_score'] = conf
else:
    cdata.obs['predicted_labels'] = cdata.obs['predicted_labels_crc'].astype('category')
    cdata.obs['conf_score'] = cdata.obs['conf_score_crc']

cdata.obs['predicted_labels'].value_counts().head(20)

### B.1 · Embedding and clustering

Standard scanpy: scale → PCA → neighbours → UMAP → Leiden, so we can visualise cell types and clusters side by side.

In [ ]:
emb = cdata[:, cdata.var["highly_variable"]].copy()
sc.pp.scale(emb, max_value=10)
sc.pp.pca(emb, use_highly_variable=True)
sc.pp.neighbors(emb)
sc.tl.umap(emb)
sc.tl.leiden(emb, resolution=2.0, key_added='leiden')

# carry embedding + clusters back onto cdata
cdata.obsm['X_umap'] = emb.obsm['X_umap']
cdata.obs['leiden'] = emb.obs['leiden'].values
sc.pl.umap(cdata, color=['leiden', 'predicted_labels'], wspace=0.4,
           legend_fontsize=6, frameon=False)

In [ ]:
# cell types in tissue space (bin2cell centroids, on the sharp H&E)
sc.set_figure_params(dpi=100, figsize=[8, 8])
sc.pl.spatial(cdata, color='predicted_labels', img_key="0.3_mpp",
              basis="spatial_cropped", spot_size=0.75, legend_fontsize=6, frameon=False)

In [ ]:
# 💾 Save annotated checkpoint
cdata.write_h5ad(f"{CKPT_DIR}/crc_b2c_annotated.h5ad")
print("saved", f"{CKPT_DIR}/crc_b2c_annotated.h5ad")

---
# Part C · Spatial proximity analysis with squidpy  *(new)*

Now we ask **which cell types sit next to which** — the part not covered in the original `bin2cell` analysis. We use two standard `squidpy` tools on the annotated single-cell object:

* **Neighbourhood enrichment** (`sq.gr.nhood_enrichment`): are two cell types found as spatial neighbours **more or less often than expected** by chance (permutation z-score)?
* **Co-occurrence** (`sq.gr.co_occurrence`): how does the probability of finding cell type B near cell type A **change with distance**?

Because `bin2cell` cells are irregularly placed (not a grid), we build a **generic** spatial graph from the cell centroids.


### ⏩ Load checkpoint (start here for Part C)

In [ ]:
import squidpy as sq
print("squidpy", sq.__version__)

cdata = sc.read_h5ad(f"{CKPT_DIR}/crc_b2c_annotated.h5ad")

# clean up: drop rare labels so the enrichment matrix is readable
cdata.obs['predicted_labels'] = cdata.obs['predicted_labels'].astype('category')
counts = cdata.obs['predicted_labels'].value_counts()
keep = counts[counts >= 50].index
cdata = cdata[cdata.obs['predicted_labels'].isin(keep)].copy()
cdata.obs['predicted_labels'] = cdata.obs['predicted_labels'].cat.remove_unused_categories()
cdata

### C.1 · Build the spatial neighbour graph

`coord_type="generic"` with a fixed number of neighbours (`n_neighs`) connects each cell to its nearest neighbours in space. (You could instead use `delaunay=True` for a Delaunay graph, or `radius=...` for a distance cutoff.)

In [ ]:
sq.gr.spatial_neighbors(cdata, coord_type="generic", n_neighs=6)

### C.2 · Neighbourhood enrichment

A high positive z-score means the two cell types are neighbours **more often than expected**; a strong negative score means they **avoid** each other. Look for tumour–immune and stroma–tumour structure.

In [ ]:
sq.gr.nhood_enrichment(cdata, cluster_key="predicted_labels", seed=0)
sc.set_figure_params(dpi=100, figsize=[9, 9])
sq.pl.nhood_enrichment(cdata, cluster_key="predicted_labels",
                       method="average", cmap="coolwarm",
                       figsize=(9, 9))

### C.3 · Co-occurrence across distance

For a chosen anchor cell type, `sq.pl.co_occurrence` shows how the conditional probability of each other type changes as you move away from it. This is a nice way to read **spatial gradients** — e.g. an immune population enriched at the tumour margin but not the core.

> Co-occurrence is computed over pairwise distances and can be memory-heavy on very large objects. If needed, subsample first (`sc.pp.subsample(cdata, n_obs=20000, copy=False)`) or restrict the `interval`.

In [ ]:
# pick an anchor cell type that exists in your data (edit as needed)
present = list(cdata.obs['predicted_labels'].cat.categories)
anchor = next((c for c in ['CMS2', 'Colonocyte', 'Macrophage', present[0]] if c in present), present[0])
print("anchor:", anchor)

sq.gr.co_occurrence(cdata, cluster_key="predicted_labels")
sq.pl.co_occurrence(cdata, cluster_key="predicted_labels", clusters=anchor, figsize=(8, 5))

### C.4 · Where are the neighbours in space?

Sanity-check a couple of interacting types by plotting them in tissue space.

In [ ]:
# highlight up to two strongly-interacting types from the enrichment map
sc.set_figure_params(dpi=100, figsize=[8, 8])
sq.pl.spatial_scatter(cdata, color="predicted_labels", shape=None, size=4,
                      library_id=None, legend_fontsize=6)

---
## Wrap-up & exercises

You reconstructed single cells from Visium HD, annotated them with CellTypist, and quantified spatial proximity with squidpy.

**Try:**
1. Re-run neighbourhood enrichment with a **Delaunay** graph (`sq.gr.spatial_neighbors(cdata, delaunay=True)`) — do the strongest interactions survive?
2. Change the CellTypist confidence handling — how sensitive are the proximity results to annotation noise?
3. Compare co-occurrence for a **tumour** anchor vs. a **stromal** anchor.
4. Compare these single-cell results against the plain 8 µm binning (the reference `bin2cell` analysis shows CellTypist confidence improves with `bin2cell` cells).
